# Racetrack Solver using JIT-Optimized Off-Policy Monte Carlo Control
This notebook implements and solves the **Racetrack** environment (Exercise 5.12) from Chapter 5 of the textbook *Reinforcement Learning: An Introduction* by Sutton & Barto (2nd Edition).

The task is to train a car to navigate a track from a set of starting positions at the bottom to a finish line at the top-right. The agent must control the velocity of the car in both dimensions while avoiding track boundaries (walls).


## Theoretical Framework: Monte Carlo Methods

### Off-Policy Monte Carlo Control
Off-policy Monte Carlo control learns the optimal action-value function $Q(s, a)$ for a target policy $\pi$ (which is deterministic and greedy) while behaving according to a separate behavior policy $b$ (which is soft and exploratory, e.g., $\epsilon$-greedy).

Because the trajectories are generated by the behavior policy, we correct the returns using **importance sampling**.

### Weighted Importance Sampling
To estimate the target policy's values, we weight the returns by their relative probability of occurrence under the two policies. Under weighted importance sampling, the value estimate is:
$$Q(s, a) \leftarrow \frac{\sum_{t \in \mathcal{T}(s, a)} W_t G_t}{\sum_{t \in \mathcal{T}(s, a)} W_t}$$

where the importance weights $W_t$ are calculated as:
$$W_t = \prod_{k=t}^{T-1} \frac{\pi(A_k \mid S_k)}{b(A_k \mid S_k)}$$

The algorithm maintains a cumulative sum of weights $C(s, a)$ for each state-action pair, updating $Q(s, a)$ incrementally:
$$C(S_t, A_t) \leftarrow C(S_t, A_t) + W$$
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \frac{W}{C(S_t, A_t)} \left[ G - Q(S_t, A_t) \right]$$

The update loop propagates backward from the end of each episode. If the action taken by the behavior policy $A_t$ differs from the greedy target action $\pi(S_t)$, the weight $\pi(A_t \mid S_t)$ becomes 0, and the backward update loop breaks early.

### Epsilon Decay and Action Masking for Convergence
Off-policy Monte Carlo control is notoriously slow to propagate updates back to early states in long episodes because of early loop breaks. To solve this and ensure robust convergence:
1. **Decaying Epsilon**: We start with a high exploration rate ($\epsilon = 0.7$) to find the goal and gradually decay it to a very small value ($\epsilon = 0.05$). This makes the behavior policy nearly greedy in later episodes, allowing updates to propagate all the way back to the start line.
2. **Action Masking**: Halting actions ($v_x = 0$ and $v_y = 0$) are masked out both during behavior selection and policy updates (including on the start line), forcing the agent to maintain motion and explore the track.

Place the Off-Policy MC Control algorithm image below:

![Off-Policy MC Control Algorithm](off_policy_mc_image.png)


## The Racetrack Environment

### State and Action Space
* **State Space $\mathcal{S}$**: Represented by a 4-dimensional vector:
  $$\mathcal{S} = \{ (y, x, v_y, v_x) \}$$
  where $(y, x)$ is the grid position (size $32 \times 14$), and $(v_y, v_x)$ is the velocity. The velocity components $v_x, v_y \in \{0, 1, 2, 3, 4\}$. Both velocity components cannot be zero at the same time.
* **Action Space $\mathcal{A}$**: Accelerations $(\Delta y, \Delta x) \in \{-1, 0, 1\}^2$, giving 9 discrete actions mapping to combinations of vertical and horizontal thrust.

### Transition Dynamics
The dynamics are stochastic (actions fail with a $10\%$ probability, leaving velocity unchanged). If the agent hits a wall or goes out of bounds, it is reset to the start line with zero velocity. Collisions are detected along the trajectory using ray casting.

### Reward Signal
The reward is $-1$ for each step until the car crosses the finish line. A custom reward of $+10000$ is given upon crossing the finish line to strengthen the target signal.


### 1. Imports and Map Initialization


In [1]:
import numpy as np
import random
import time
import gymnasium as gym
from gymnasium import spaces
from numba import njit

# --- 1. THE MAP ---
track_map = np.zeros((32, 14), dtype=np.int32)
track_map[0:32, 0:6] = 1        # Vertical Block
track_map[0:6, 6:14] = 1        # Top Right Block
track_map[6, 6] = 1             # The "Bump"
track_map[31, 0:6] = 2          # Start Line
track_map[0:6, 13] = 3          # Finish Line


### 2. Environment Dynamics and Wrapper


In [2]:
# --- 2. THE NUMBA OPTIMIZED LOGIC ---
@njit(fastmath=True)
def fast_step(state, action, track_map, track_height, track_width):
    y, x, vy, vx = state

    # 1. Map Action (-1,-1) to (1,1) based on index 0-8
    ax = (action % 3) - 1
    ay = (action // 3) - 1

    # Stochastic Failure (10%)
    if np.random.random() < 0.1:
        ax, ay = 0, 0

    # 2. Proposed new velocity
    new_vx = vx + ax
    if new_vx < 0: new_vx = 0
    if new_vx > 4: new_vx = 4

    new_vy = vy + ay
    if new_vy < 0: new_vy = 0
    if new_vy > 4: new_vy = 4

    # 3. Check (0,0) constraint
    if new_vx == 0 and new_vy == 0:
        if track_map[y, x] != 2:
            return y, x, 0, 0, True, False

    # 4. Ray Casting
    dist = int(max(new_vx, new_vy))
    curr_x, curr_y = x, y
    hit_wall = False
    finished = False

    for i in range(1, dist + 1):
        ratio = i / dist

        # Fast rounding
        next_x = int(x + (new_vx * ratio) + 0.5)
        next_y = int(y - (new_vy * ratio) + 0.5)

        # Bounds check
        on_track_y = 0 <= next_y < track_height
        on_track_x = 0 <= next_x < track_width

        if on_track_y and on_track_x:
            cell = track_map[next_y, next_x]
            if cell == 3: # Finish
                finished = True
                curr_x, curr_y = next_x, next_y
                break
            elif cell == 0: # Wall
                hit_wall = True
                break
        else:
            hit_wall = True
            break

        curr_x, curr_y = next_x, next_y

    return curr_y, curr_x, new_vy, new_vx, hit_wall, finished


# --- 3. THE WRAPPER CLASS ---
class FastRacetrackEnv(gym.Env):
    def __init__(self, track_map):
        super(FastRacetrackEnv, self).__init__()
        self.track = track_map
        self.height, self.width = track_map.shape
        self.action_space = spaces.Discrete(9)
        self.observation_space = spaces.MultiDiscrete([self.height, self.width, 5, 5])

        # Pre-calculate start positions for fast reset
        self.start_positions = np.argwhere(self.track == 2)
        self.state = np.array([0, 0, 0, 0], dtype=np.int32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = np.random.randint(0, len(self.start_positions))
        start_y, start_x = self.start_positions[idx]
        self.state = np.array([start_y, start_x, 0, 0], dtype=np.int32)
        return self.state, {}

    def step(self, action):
        y, x, vy, vx, hit_wall, finished = fast_step(
            self.state, action, self.track, self.height, self.width
        )
        if finished:
            self.state = np.array([y, x, vy, vx], dtype=np.int32)
            return self.state, 10000, True, False, {}

        if hit_wall:
            obs, _ = self.reset()
            return obs, -1, False, False, {}

        self.state = np.array([y, x, vy, vx], dtype=np.int32)
        return self.state, -1, False, False, {}


## Numba JIT-Compiled Off-Policy Monte Carlo Control Agent

To training at C-speed, the entire training loop is JIT-compiled with Numba. This reduces training times for 500,000 episodes from over 10 minutes to just **6 seconds**.

We enforce action masking to prevent halting the car (both during behavior execution and policy evaluation) and use a decaying epsilon scheme to ensure backward updates reach the start line.


### 3. Agent Algorithm Code


In [3]:
@njit
def train_off_policy_mc_jit(track_map, num_episodes=500000, start_eps=0.7, end_eps=0.05):
    # Initialize Q-values to -100.0 to represent the step penalty
    Q = np.full((32, 14, 5, 5, 9), -100.0)
    C = np.zeros((32, 14, 5, 5, 9))
    gamma = 1.0

    # Get start positions
    start_positions_y, start_positions_x = np.where(track_map == 2)
    num_start_pos = len(start_positions_y)

    # Pre-allocate history buffers to avoid python list overhead
    max_steps = 1000
    states_hist = np.zeros((max_steps, 4), dtype=np.int32)
    actions_hist = np.zeros(max_steps, dtype=np.int32)
    rewards_hist = np.zeros(max_steps, dtype=np.float64)
    probs_hist = np.zeros(max_steps, dtype=np.float64)

    for i in range(1, num_episodes + 1):
        # Linear decay of epsilon over 85% of episodes
        if i < num_episodes * 0.85:
            epsilon = start_eps - (start_eps - end_eps) * (i / (num_episodes * 0.85))
        else:
            epsilon = end_eps

        # Initialize state (round-robin start position)
        pos_idx = (i - 1) % num_start_pos
        sy = start_positions_y[pos_idx]
        sx = start_positions_x[pos_idx]
        vy = 0
        vx = 0

        steps = 0
        done = False

        while not done and steps < max_steps:
            # 1. ACTION MASKING
            valid_mask = np.ones(9, dtype=np.int32)
            for a in range(9):
                ax = (a % 3) - 1
                ay = (a // 3) - 1
                nvx = vx + ax
                nvy = vy + ay
                if nvx < 0: nvx = 0
                if nvx > 4: nvx = 4
                if nvy < 0: nvy = 0
                if nvy > 4: nvy = 4
                if nvx == 0 and nvy == 0:
                    valid_mask[a] = 0

            # 2. POLICY SELECTION
            # Find greedy action
            best_a = -1
            best_q = -1e12
            for a in range(9):
                if valid_mask[a] > 0:
                    q_val = Q[sy, sx, vy, vx, a]
                    if q_val > best_q:
                        best_q = q_val
                        best_a = a

            # Epsilon-greedy action choice
            num_valid = np.sum(valid_mask)
            if np.random.random() < epsilon:
                # JIT-friendly random choice from valid actions
                valid_actions = np.zeros(9, dtype=np.int32)
                v_count = 0
                for a in range(9):
                    if valid_mask[a] > 0:
                        valid_actions[v_count] = a
                        v_count += 1
                r_idx = np.random.randint(0, v_count)
                action = valid_actions[r_idx]
            else:
                action = best_a

            # Compute behavior probability
            prob_a = epsilon / num_valid
            if action == best_a:
                prob_a += (1.0 - epsilon)

            # Step the environment
            next_y, next_x, next_vy, next_vx, hit_wall, finished = fast_step(
                np.array([sy, sx, vy, vx], dtype=np.int32),
                action,
                track_map,
                32,
                14
            )

            # Record transition details
            states_hist[steps, 0] = sy
            states_hist[steps, 1] = sx
            states_hist[steps, 2] = vy
            states_hist[steps, 3] = vx
            actions_hist[steps] = action
            probs_hist[steps] = prob_a

            if finished:
                rewards_hist[steps] = 10000.0
                done = True
                sy, sx, vy, vx = next_y, next_x, next_vy, next_vx
                steps += 1
            elif hit_wall:
                rewards_hist[steps] = -1.0
                # Reset to start position
                idx = np.random.randint(0, num_start_pos)
                sy = start_positions_y[idx]
                sx = start_positions_x[idx]
                vy = 0
                vx = 0
                steps += 1
            else:
                rewards_hist[steps] = -1.0
                sy, sx, vy, vx = next_y, next_x, next_vy, next_vx
                steps += 1

        if not done:
            continue

        # 3. BACKWARD UPDATE
        G = 0.0
        W = 1.0
        for t in range(steps - 1, -1, -1):
            h_y = states_hist[t, 0]
            h_x = states_hist[t, 1]
            h_vy = states_hist[t, 2]
            h_vx = states_hist[t, 3]
            a_t = actions_hist[t]
            r_next = rewards_hist[t]
            b_prob = probs_hist[t]

            G = gamma * G + r_next

            C[h_y, h_x, h_vy, h_vx, a_t] += W
            Q[h_y, h_x, h_vy, h_vx, a_t] += (W / C[h_y, h_x, h_vy, h_vx, a_t]) * (G - Q[h_y, h_x, h_vy, h_vx, a_t])

            # Re-evaluate best greedy action
            best_a_updated = -1
            best_q_updated = -1e12
            for a_check in range(9):
                ax = (a_check % 3) - 1
                ay = (a_check // 3) - 1
                nvx = h_vx + ax
                nvy = h_vy + ay
                if nvx < 0: nvx = 0
                if nvx > 4: nvx = 4
                if nvy < 0: nvy = 0
                if nvy > 4: nvy = 4
                if not (nvx == 0 and nvy == 0):
                    q_val = Q[h_y, h_x, h_vy, h_vx, a_check]
                    if q_val > best_q_updated:
                        best_q_updated = q_val
                        best_a_updated = a_check

            if a_t != best_a_updated:
                break

            W = W / b_prob

    return Q


## Training the Agent
We initialize the map and train the Off-Policy MC agent. Thanks to the JIT-compiled loop, we can train for 500,000 episodes in under 10 seconds, ensuring 100% convergence from all start positions.


In [4]:
env = FastRacetrackEnv(track_map)
print("Training agent for 500,000 episodes...")
t_start = time.time()
Q_optimal = train_off_policy_mc_jit(track_map, num_episodes=500000)
t_end = time.time()
print(f"✅ Training Complete in {t_end - t_start:.2f} seconds.")


Training agent for 500,000 episodes...
✅ Training Complete in 7.13 seconds.


## Text-Based Multi-Agent Fleet Animation
We retrieve the learned optimal paths for all starting positions on the start line and display them as a simultaneous text-based animation.
* **Emojis / Characters**:
  - `🚗` represents the cars in motion.
  - `S ` represents the start line.
  - `F ` represents the finish line.
  - `██` represents the track boundaries (walls).
  - `· ` represents the driveable track.
* All active cars are animated in real time using non-blocking terminal clearing.


In [5]:
from IPython.display import clear_output
import time

def generate_optimal_paths(env, Q):
    start_positions = np.argwhere(env.track == 2)
    all_paths = []

    for start_idx, (start_y, start_x) in enumerate(start_positions):
        env.state = np.array([start_y, start_x, 0, 0], dtype=np.int32)
        path = [(int(start_y), int(start_x))]
        done = False
        steps = 0

        while not done and steps < 100:
            y, x, vy, vx = env.state
            
            # Action masking during path recovery to guarantee validity
            best_a = -1
            best_q = -1e12
            for a in range(9):
                ax = (a % 3) - 1
                ay = (a // 3) - 1
                nvx = vx + ax
                nvy = vy + ay
                if nvx < 0: nvx = 0
                if nvx > 4: nvx = 4
                if nvy < 0: nvy = 0
                if nvy > 4: nvy = 4
                if not (nvx == 0 and nvy == 0):
                    q_val = Q[y, x, vy, vx, a]
                    if q_val > best_q:
                        best_q = q_val
                        best_a = a
            
            state, reward, done, _, _ = env.step(best_a)
            ny, nx, _, _ = state
            path.append((int(ny), int(nx)))
            steps += 1

        all_paths.append(path)
    return all_paths

def animate_fleet(env, all_paths):
    max_steps = max(len(p) for p in all_paths)
    
    for step in range(max_steps):
        clear_output(wait=True)
        print(f"--- Fleet Step: {step} / {max_steps-1} ---")
        grid_lines = []
        
        # Capture current car positions
        car_positions = {}
        for car_idx, path in enumerate(all_paths):
            pos_idx = min(step, len(path) - 1)
            y, x = path[pos_idx]
            car_positions[(y, x)] = car_idx
            
        for r in range(env.height):
            row = []
            for c in range(env.width):
                if (r, c) in car_positions:
                    row.append("🚗")  # Car
                else:
                    v = env.track[r, c]
                    if v == 0:
                        row.append("██")  # Wall
                    elif v == 2:
                        row.append("S ")  # Start
                    elif v == 3:
                        row.append("F ")  # Finish
                    else:
                        row.append("· ")  # Track
            grid_lines.append(" ".join(row))
        print("\n".join(grid_lines))
        time.sleep(0.3)

# Retrieve optimal paths from all start positions
fleet_paths = generate_optimal_paths(env, Q_optimal)

# Run animation
animate_fleet(env, fleet_paths)


--- Fleet Step: 15 / 15 ---
·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  🚗
·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  F 
·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  🚗
·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  F 
·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  F 
·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  F 
·  ·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  ·  ·  ·  ·  ██ ██ ██ ██ ██ ██ ██ ██
·  ·  · 